# Data Cleaning & Processing (Pipeline Overview)

This notebook demonstrates how the processed dataset is built using a reusable Python script.

Instead of performing all transformations inside the notebook, we use a production-style pipeline:

- Load raw data
- Clean and standardize data
- Merge datasets
- Create business features
- Export processed dataset

The goal is to make the pipeline:
- reproducible
- modular
- easier to maintain

In [4]:
import os
import sys
import pandas as pd

In [5]:
# Ensure project root is in path (important if running notebook from /notebook folder)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Project root: x:\Project\olist-ecommerce-data-analysis


##  Data Processing Pipeline

The processed dataset is built using the script:

`scripts/build_processed_data.py`

### Key Steps:

1. **Load raw datasets**
   - orders, items, payments, reviews, customers, products, sellers

2. **Data Cleaning**
   - Remove duplicates
   - Convert data types (timestamps, numeric)
   - Handle missing values

3. **Aggregation**
   - Payment aggregation per order
   - Review aggregation per order

4. **Merging**
   - Combine all datasets into a single analytical table

5. **Feature Engineering**
   - Revenue (item + shipping)
   - Delivery metrics (delivery_days, delay)
   - Time features (month, weekday, hour)
   - Flags (late delivery, missing values)

6. **Output**
   - CSV: `data/processed/clean_olist_data.csv`
   - SQLite DB: `data/processed/olist_analysis.db`

In [6]:
# Run the processing script

script_path = os.path.join(PROJECT_ROOT, "scripts", "build_processed_data.py")

print("Running script:", script_path)
os.system(f"python {script_path}")

Running script: x:\Project\olist-ecommerce-data-analysis\scripts\build_processed_data.py


1

## 📦 Processed Dataset Output

After running the script, the processed dataset is saved to:

- `data/processed/clean_olist_data.csv`
- `data/processed/olist_analysis.db`

This dataset is **analysis-ready** and includes:

- Order-level + item-level information
- Customer and seller attributes
- Product categories
- Payment and review aggregates
- Engineered business features

In [7]:
# Load processed dataset

processed_path = os.path.join(PROJECT_ROOT, "data", "processed", "clean_olist_data.csv")

df = pd.read_csv(processed_path)

print("Dataset shape:", df.shape)

Dataset shape: (112650, 44)


In [11]:
print("Dataset shape:", df.shape)
print("Unique orders:", df["order_id"].nunique())
print("Unique customers:", df["customer_unique_id"].nunique())
print("Rows per order (avg):", round(len(df) / df["order_id"].nunique(), 2))

print("\nImportant modeling note:")
print("- The processed dataset is at order-item level")
print("- 1 row = 1 item in an order")
print("- Business KPIs should be calculated at order level, not directly from raw rows")

Dataset shape: (112650, 44)
Unique orders: 98666
Unique customers: 95420
Rows per order (avg): 1.14

Important modeling note:
- The processed dataset is at order-item level
- 1 row = 1 item in an order
- Business KPIs should be calculated at order level, not directly from raw rows


In [13]:
print(
    """
Payment modeling note:
- order_total_payment_value is an order-level field merged into an item-level table
- the same payment total is repeated across all item rows of the same order
- do not sum order_total_payment_value directly from item-level rows
"""
)


Payment modeling note:
- order_total_payment_value is an order-level field merged into an item-level table
- the same payment total is repeated across all item rows of the same order
- do not sum order_total_payment_value directly from item-level rows



In [8]:
df.head()

,order_id,order_item_id,product_id,seller_id,customer_id,customer_unique_id,customer_city,customer_state,seller_city,seller_state,...,order_total_payment_value,review_score,review_count,has_review_comment,missing_review_score_flag,product_category_name,product_category_name_english,missing_product_category_flag,missing_delivery_info_flag,order_status_delivery_conflict_flag
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,volta redonda,SP,...,72.19,5.0,1,1,0,cool_stuff,cool_stuff,0,0,0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,sao paulo,SP,...,259.83,4.0,1,0,0,pet_shop,pet_shop,0,0,0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,borda da mata,MG,...,216.87,5.0,1,1,0,moveis_decoracao,furniture_decor,0,0,0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,atibaia,SP,franca,SP,...,25.78,4.0,1,0,0,perfumaria,perfumery,0,0,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,varzea paulista,SP,loanda,PR,...,218.04,5.0,1,1,0,ferramentas_jardim,garden_tools,0,0,0


In [9]:
df.columns.tolist()

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'customer_id',
 'customer_unique_id',
 'customer_city',
 'customer_state',
 'seller_city',
 'seller_state',
 'order_status',
 'is_delivered',
 'is_canceled',
 'order_purchase_timestamp',
 'order_date',
 'order_month',
 'order_year',
 'order_quarter',
 'order_weekday',
 'purchase_hour',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'estimated_delivery_days',
 'delivery_delay_days',
 'is_late_delivery',
 'delivery_status',
 'price',
 'freight_value',
 'item_revenue',
 'shipping_revenue',
 'revenue',
 'payment_installments_max',
 'payment_type_nunique',
 'order_total_payment_value',
 'review_score',
 'review_count',
 'has_review_comment',
 'missing_review_score_flag',
 'product_category_name',
 'product_category_name_english',
 'missing_product_category_flag',
 'missing_delivery_info_flag',
 'order_status_delivery_conflict_flag']

## 🧠 Key Features in Processed Dataset

### Revenue
- `item_revenue` = product price
- `shipping_revenue` = freight value
- `revenue` = total revenue per row

### Delivery Metrics
- `delivery_days` = actual delivery time
- `delivery_delay_days` = delay vs estimated
- `is_late_delivery` = 1 if delayed

### Time Features
- `order_month`
- `order_year`
- `order_weekday`
- `purchase_hour`

### Customer & Product
- `customer_unique_id`
- `product_category_name_english`

### Data Quality Flags
- `missing_review_score_flag`
- `missing_product_category_flag`
- `missing_delivery_info_flag`

These features are used for:
- KPI calculation
- dashboard building
- deeper analysis (RFM, delivery impact)

In [14]:
order_level_check = (
    df.sort_values(["order_id", "order_item_id"])
    .groupby("order_id", as_index=False)
    .agg(
        is_delivered=("is_delivered", "max"),
        order_revenue=("revenue", "sum"),
        review_score=("review_score", "first"),
        is_late_delivery=("is_late_delivery", "max"),
    )
)

delivered_orders = order_level_check[order_level_check["is_delivered"] == 1].copy()

print("Missing review score rate (item-level rows):", round(df["review_score"].isna().mean(), 4))
print("Delivered orders:", delivered_orders["order_id"].nunique())
print("Delivered revenue (order-level):", round(delivered_orders["order_revenue"].sum(), 2))
print("Average review score (order-level):", round(delivered_orders["review_score"].mean(), 4))
print("Late delivery rate (order-level):", round(delivered_orders["is_late_delivery"].mean(), 4))

Missing review score rate (item-level rows): 0.0
Delivered orders: 96478
Delivered revenue (order-level): 15419773.75
Average review score (order-level): 4.1427
Late delivery rate (order-level): 0.0677


## Quick Observations

- The processed dataset is intentionally stored at the **order-item level**.
- Revenue at row level is valid for item/category analysis, but business KPIs should be calculated at the **order level**.
- `order_total_payment_value` is duplicated across item rows after merge and must be handled carefully.
- Delivered-order filtering should be used for primary revenue KPIs and performance reporting.

This processed layer is now ready for:
- SQL analysis
- dashboard modeling
- order-level KPI aggregation
- advanced customer analytics